In [ ]:
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import pandas as pd
import numpy as np
import warnings
import pickle

warnings.filterwarnings("ignore")

In [ ]:
pickled_model = pickle.load(open('xgb_model.pkl', 'rb'))

In [ ]:
added_products = pd.read_csv("added_products.csv")

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(added_products.added_product)
added_products['encoded_products'] = label_encoder.\
    transform(added_products['added_product'])

In [ ]:
products = ["ind_ahor_fin_ult1", "ind_aval_fin_ult1", "ind_cco_fin_ult1",
            "ind_cder_fin_ult1", "ind_cno_fin_ult1",  "ind_ctju_fin_ult1", 
            "ind_ctma_fin_ult1", "ind_ctop_fin_ult1", "ind_ctpp_fin_ult1",
            "ind_deco_fin_ult1", "ind_deme_fin_ult1", "ind_dela_fin_ult1", 
            "ind_ecue_fin_ult1", "ind_fond_fin_ult1", "ind_hip_fin_ult1",
            "ind_plan_fin_ult1", "ind_pres_fin_ult1", "ind_reca_fin_ult1", 
            "ind_tjcr_fin_ult1", "ind_valo_fin_ult1", "ind_viv_fin_ult1",
            "ind_nomina_ult1",   "ind_nom_pens_ult1", "ind_recibo_ult1"]

In [ ]:
test_data_2016 = pd.read_csv("data/test_dataset_new.csv")

In [ ]:
test_data_2016.info()

In [ ]:
for col in products:
    test_data_2016[col] = [0] * test_data_2016.shape[0]

In [ ]:
test_data_2016.info()

In [ ]:
def process_data_from_original_dataframe(combined, encoders=None):
    #fix sexo
    combined['sexo'] = combined['sexo'].fillna("V")
    
    #fix ind_empleado
    combined['ind_empleado'] = combined['ind_empleado'].fillna("N")
    
    #fix segmento
    combined["segmento"] = combined["segmento"].fillna("02 - PARTICULARES")
    
    # fixing age
    combined['age'] = pd.to_numeric(combined['age'], errors='coerce')

    combined.loc[combined.age < 18, "age"] = \
        combined.loc[(combined.age > 18) &
                     (combined.age <= 30), "age"].mean(skipna=True)
    combined.loc[combined.age > 100, "age"] = \
        combined.loc[(combined.age > 30) &
                     (combined.age <= 100), "age"].mean(skipna=True)
    combined['age'].fillna(combined['age'].mean(), inplace=True)
    combined['age'] = combined['age'].astype(int)

    # fix ind_nuevo
    combined.loc[combined.ind_nuevo.isnull(), 'ind_nuevo'] = 1

    # fix antiguedad
    combined['antiguedad'] = pd.to_numeric(combined['antiguedad'],
                                           errors='coerce')
    combined.loc[combined.antiguedad.isnull(), 'antiguedad'] = \
        combined.antiguedad.min()
    combined.loc[combined.antiguedad < 0, 'antiguedad'] = 0

    # fix indrel
    combined.loc[combined.indrel.isnull(), 'indrel'] = 1

    # drop useless cols
    combined.drop(['tipodom', 'cod_prov'], axis=1, inplace=True)

    # fix ind_actividad_cliente
    combined['ind_actividad_cliente'] = pd.to_numeric(combined['ind_actividad_cliente'],
                                           errors='coerce')
    combined.loc[combined.ind_actividad_cliente.isnull(),
                 "ind_actividad_cliente"] = \
        combined.ind_actividad_cliente.median()

    # fix city name
    combined.loc[combined.nomprov ==
                 "CORU\xc3\x91A, A", "nomprov"] = "CORUNA, A"
    combined.loc[combined.nomprov.isnull(), 'nomprov'] = 'UNKNOWN'

    # fix incomes
    combined['renta'] = pd.to_numeric(combined['renta'], errors='coerce')

    grouped = combined.groupby('nomprov').\
        agg({'renta': lambda x: x.median(skipna=True)}).reset_index()
    new_incomes = pd.merge(combined, grouped,
                           how='inner',
                           on='nomprov').loc[:, ['nomprov', 'renta_y']]

    new_incomes = new_incomes.\
        rename(columns={"renta_y": "renta"}).\
        sort_values("renta").sort_values("nomprov")

    combined.sort_values("nomprov", inplace=True)
    combined = combined.reset_index()
    new_incomes = new_incomes.reset_index()
    combined.loc[combined.renta.isnull(), "renta"] = \
        new_incomes.loc[combined.renta.isnull(), "renta"].median()
    combined.sort_values(by='fecha_dato', inplace=True)

    # rest of the columns
    string_data = combined.select_dtypes(include=["object"])
    missing_columns = [col for col in string_data
                       if string_data[col].isnull().any()]
    del string_data

    combined.loc[combined.indfall.isnull(), 'indfall'] = 'N'
    combined.loc[combined.tiprel_1mes.isnull(), 'tiprel_1mes'] = 'A'
    combined.tiprel_1mes = combined.tiprel_1mes.astype('category')

    map_dict = {
        '1.0': '1',
        '1': '1',
        '3.0': '3',
        'P': 'P',
        3.0: '3',
        2.0: '2',
        '3': '3',
        '2.0': '2',
        '4.0': '4',
        '4': '4',
        '2': '2',
        1.0: '1',
        4.0: '4'
    }

    combined.indrel_1mes.fillna('P', inplace=True)
    combined.indrel_1mes = combined.indrel_1mes.apply(lambda x: map_dict[x])
    combined.indrel_1mes = combined.indrel_1mes.astype('category')

    unknown_cols = [col for col in missing_columns if
                    col not in ['indfall', 'tiprel_1mes', 'indrel_1mes']]
    for col in unknown_cols:
        combined.loc[combined[col].isnull(), col] = "UNKNOWN"

    # feature cols
    feature_cols = combined.iloc[:1, ].filter(regex="ind_+.*ult.*").\
        columns.values
    for col in feature_cols:
        combined.loc[combined[col].isnull(), col] = 0
        combined[col] = combined[col].astype(int, errors='ignore')

    del combined['ult_fec_cli_1t'], combined['fecha_alta']
    encoder_cols = ['sexo', 'indrel_1mes', 'pais_residencia', 'ind_empleado',
                'segmento', 'tiprel_1mes', 'indresi', 'indext', 'conyuemp',
                'canal_entrada', 'indfall', 'nomprov']
    if encoders == None:
        encoders = []
        for col in encoder_cols:
            temp_enc = LabelEncoder()
            temp_enc.fit(combined[col])
            combined[col] = temp_enc.transform(combined[col])
            encoders.append(temp_enc)
    else:
        for i in range(len(encoders)):
            combined[encoder_cols[i]] = encoders[i].transform(combined[encoder_cols[i]])
            #print(combined[encoder_cols[i]])
        
    return combined, encoders

In [ ]:
combined, encoders = process_data_from_original_dataframe(test_data_2016)

In [ ]:
combined = combined.reset_index(drop=True).drop(["index", "fecha_dato"], axis=1)

In [ ]:
rename_columns = {
    "ind_empleado": "employee_index",
    "pais_residencia": "country_residence",
    "sexo": "sex",
    "age": "age",
    "ind_nuevo": "new_customer_index",
    "antiguedad": "seniority",
    "indrel": "primary_customer",
    "indrel_1mes": "customer_type_beginning",
    "tiprel_1mes": "customer_relation_type_beginning",
    "indresi": "residence_index",
    "indext": "foreigner_index",
    "conyuemp": "spouse_index",
    "canal_entrada": "channel",
    "indfall": "deceased_index",
    "tipodom": "address_type",
    "cod_prov": "province_code",
    "nomprov": "province_name",
    "ind_actividad_cliente": "activity_index",
    "renta": "income",
    "segmento": "segmentation",
    "ind_ahor_fin_ult1": "saving_account",
    "ind_aval_fin_ult1": "guarantees",
    "ind_cco_fin_ult1": "current_accounts",
    "ind_cder_fin_ult1": "derivada_account",
    "ind_cno_fin_ult1": "payroll_account",
    "ind_ctju_fin_ult1": "junior_account",
    "ind_ctma_fin_ult1": "mas_particular_account",
    "ind_ctop_fin_ult1": "particular_account",
    "ind_ctpp_fin_ult1": "particular_plus_account",
    "ind_deco_fin_ult1": "short_term_deposits",
    "ind_deme_fin_ult1": "medium_term_deposits",
    "ind_dela_fin_ult1": "long_term_deposits",
    "ind_ecue_fin_ult1": "e_account",
    "ind_fond_fin_ult1": "funds",
    "ind_hip_fin_ult1": "mortgage",
    "ind_plan_fin_ult1": "pensions",
    "ind_pres_fin_ult1": "loans",
    "ind_reca_fin_ult1": "taxes",
    "ind_tjcr_fin_ult1": "credit_card",
    "ind_valo_fin_ult1": "securities",
    "ind_viv_fin_ult1": "home_account",
    "ind_nomina_ult1": "payroll",
    "ind_nom_pens_ult1": "pensions",
    "ind_recibo_ult1": "direct_debit",
}

In [ ]:
combined = combined.rename(rename_columns, axis=1)

In [ ]:
combined = combined.loc[:,~combined.columns.duplicated()]

In [ ]:
combined.drop("ncodpers", axis=1, inplace=True)
combined["month"] = [6] * combined.shape[0]

In [ ]:
combined.head()

In [ ]:
combined.to_csv("processed_test.csv", index=False)

In [ ]:
X_test = combined.sample(1)

final_test = xgb.DMatrix(X_test)
preds = pickled_model.predict(final_test)

In [ ]:
preds[0]

In [ ]:
np.argsort(preds[0], axis=0)[::-1][:7]

In [ ]:
top_t_products = [label_encoder.inverse_transform(np.argsort(pred, axis=0)[::-1][:7]) for pred in preds]

In [ ]:
top_t_products